# 00 - Environment setup

Run this once per machine or pod. It checks the GPU, verifies the environment,
runs the test suite, and pre-downloads the pretrained models so later notebooks
never stall mid-run.

## Creating the environment (shell, once)

```bash
conda create -n adaptts python=3.11 -y
conda activate adaptts

# CUDA build of PyTorch. cu121 works on Turing (GTX 16xx) through Ada (4090).
pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu121

pip install -r requirements.txt
python -m ipykernel install --user --name adaptts --display-name "AdapTTS (conda)"
```

Then pick the **AdapTTS (conda)** kernel in Jupyter before running anything
below. The first cell prints which interpreter you are actually on, so a
wrong-kernel mistake shows up immediately rather than as a confusing import
error later.

## The second environment: CATT

The diacritizer pins an older torch and needs `pytorch_lightning`, which does
not coexist cleanly with this project's pins. It therefore gets its own
environment, used by exactly one step in notebook 01 and never again.

```bash
conda create -n CATT python=3.11 -y
conda activate CATT
pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu121
pip install pytorch-lightning num2words tqdm pyyaml
conda deactivate
```

Upload the `catt_tashkeel` folder so it sits at `/workspace/catt_parent/catt_tashkeel`,
and set `paths.catt_root: /workspace/catt_parent` in your config. Only the ECA
checkpoint is needed; the MSA weights are never loaded.

Expected: about 5 minutes plus roughly 4 GB of model downloads.

In [ ]:
!nvidia-smi

In [ ]:
import sys, os

print("interpreter:", sys.executable)
print("python     :", sys.version.split()[0])
in_conda = "adaptts" in sys.executable.lower() or os.environ.get("CONDA_DEFAULT_ENV") == "adaptts"
print("env        :", os.environ.get("CONDA_DEFAULT_ENV", "(none)"))
if not in_conda:
    print()
    print("WARNING: this does not look like the adaptts environment.")
    print("Select the 'AdapTTS (conda)' kernel from the kernel picker.")

In [ ]:
import os, sys
REPO = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, "src"))
os.environ["PYTHONIOENCODING"] = "utf-8"
print("repo:", REPO)

## Verify dependencies

If anything is missing here, run the pip commands from the shell block above in
a terminal, then restart the kernel.

In [ ]:
import importlib

required = [
    ("torch", "torch"), ("torchaudio", "torchaudio"), ("transformers", "transformers"),
    ("numpy", "numpy"), ("scipy", "scipy"), ("soundfile", "soundfile"),
    ("pyarrow", "pyarrow"), ("yaml", "pyyaml"), ("tqdm", "tqdm"),
    ("tensorboard", "tensorboard"), ("huggingface_hub", "huggingface_hub"),
]
missing = []
for mod, pkg in required:
    try:
        m = importlib.import_module(mod)
        print(f"  ok      {pkg:<18} {getattr(m, '__version__', '')}")
    except ImportError:
        missing.append(pkg)
        print(f"  MISSING {pkg}")
if missing:
    raise SystemExit("install these first: pip install " + " ".join(missing))

In [ ]:
import torch

print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print()
    print("No GPU visible. Training will run, but very slowly.")
else:
    p = torch.cuda.get_device_properties(0)
    cc = p.major + p.minor / 10
    print(f"gpu     : {p.name}")
    print(f"memory  : {p.total_memory / 1024 ** 3:.1f} GB")
    print(f"compute : {p.major}.{p.minor}")
    print()
    # torch reports is_bf16_supported() True on Turing, but that is emulation,
    # not hardware. Only Ampere and newer have bf16 tensor cores.
    if cc >= 8.0:
        print("Ampere or newer: use train.precision = bf16 (no gradient scaler needed).")
        print("  -> configs/exp1_egyptian.yaml is already set up this way.")
    else:
        print("Turing or older: NO hardware bf16, despite what torch reports.")
        print("Use train.precision = fp16 with a gradient scaler.")
        print("  -> configs/exp0_small.yaml is already set up this way.")
    if p.total_memory / 1024 ** 3 < 10:
        print()
        print("Under 10 GB: start with configs/exp0_small.yaml.")
    x = torch.randn(1024, 1024, device="cuda")
    torch.cuda.synchronize()
    print()
    print("GPU matmul check:", bool(torch.isfinite((x @ x).sum())))

## Run the test suite

These check causality, KV-cache equivalence, collation alignment and, most
importantly, that the model actually learns homograph disambiguation on a
controlled corpus. All must pass before you spend GPU time.

In [ ]:
import subprocess, sys, os

tests = [
    "tests/test_egyptian.py",     # text normalization, the waw rule
    "tests/test_models.py",       # causality, KV cache, masking
    "tests/test_data.py",         # collation, bucketing, config
    "tests/test_end_to_end.py",   # does it actually learn to disambiguate
    "tests/test_integration.py",  # the real pipeline on synthetic data
]
env = dict(os.environ, PYTHONIOENCODING="utf-8")
for t in tests:
    print()
    print("=" * 60)
    print(t)
    print("=" * 60)
    r = subprocess.run([sys.executable, t], capture_output=True, text=True, env=env)
    print(r.stdout[-2500:])
    if r.returncode != 0:
        print("STDERR:", r.stderr[-2000:])
        raise SystemExit(t + " FAILED - fix this before continuing")
print()
print("All tests passed. The code is ready to train.")

## Check the Egyptian text normalizer

This runs its own suite. The headline rule: no linking waw between magnitude
groups, so 2024 is "الفين اربعة و عشرين", never "الفين و اربعة و عشرين".

In [ ]:
!python src/adaptts/text/egyptian.py

## Pre-download the pretrained models

In [ ]:
import yaml
from transformers import (
    AutoFeatureExtractor, AutoModel, AutoModelForCTC, AutoProcessor, AutoTokenizer, MimiModel,
)

cfg = yaml.safe_load(open("configs/base.yaml", encoding="utf-8"))

print("1/4 CTC aligner")
AutoProcessor.from_pretrained(cfg["align"]["model_id"])
AutoModelForCTC.from_pretrained(cfg["align"]["model_id"])

print("2/4 SSL span encoder (analysis only; labels no longer use it)")
AutoFeatureExtractor.from_pretrained(cfg["spanemb"]["model_id"])
AutoModel.from_pretrained(cfg["spanemb"]["model_id"])

print("3/4 MARBERTv2 teacher")
AutoTokenizer.from_pretrained(cfg["teacher"]["model_id"])
AutoModel.from_pretrained(cfg["teacher"]["model_id"])

print("4/4 Mimi codec")
MimiModel.from_pretrained(cfg["codec_model_id"])

print()
print("all models cached")

## Check the CATT environment

The diacritizer runs in its own environment, so it cannot be imported here.
This checks it the way notebook 01 will actually invoke it: a subprocess under
the `CATT` interpreter that loads the ECA checkpoint and diacritizes two probe
sentences.

Catching a broken CATT setup now costs a minute. Catching it in notebook 01
costs the 40 minutes of alignment you already paid for.

In [ ]:
import os, subprocess, sys, yaml

_cfg = yaml.safe_load(open("configs/base.yaml", encoding="utf-8"))
CATT_ROOT = os.environ.get("CATT_ROOT") or _cfg["paths"].get("catt_root", "")
CATT_PY = os.environ.get(
    "CATT_PY", os.path.expanduser("~/miniconda3/envs/CATT/bin/python")
)

print("catt_root:", CATT_ROOT or "(not set)")
print("catt python:", CATT_PY)

ok = True
if not CATT_ROOT or not os.path.isdir(os.path.join(CATT_ROOT, "catt_tashkeel")):
    ok = False
    print()
    print("No catt_tashkeel/ under catt_root.")
    print("Upload the folder and set paths.catt_root in your config.")
elif not os.path.isfile(CATT_PY):
    ok = False
    print()
    print("CATT interpreter not found. Create the env (see the shell block above),")
    print("or set CATT_PY to its python.")
else:
    ckpt = os.path.join(CATT_ROOT, "catt_tashkeel", "checkpoints", "eca_model_weights.pt")
    print("eca checkpoint:", "found" if os.path.isfile(ckpt) else "MISSING")
    probe = (
        "import sys; sys.path.insert(0, %r); sys.path.insert(0, 'scripts')\n"
        "from diacritize import load_eca_model\n"
        "m, pre, post, tok = load_eca_model(%r)\n"
        "t = ['انا شوفت علم مصر بيرفرف', 'علم الفيزيا من اهم العلوم']\n"
        "p = [pre.process_text(tok.remove_tashkeel(x), verbose=False) for x in t]\n"
        "o = [post.process(x) for x in m.do_tashkeel_batch(p, batch_size=2, verbose=False)]\n"
        "print(o[0]); print(o[1])\n"
    ) % (CATT_ROOT, CATT_ROOT)
    r = subprocess.run([CATT_PY, "-c", probe], capture_output=True, text=True,
                       env=dict(os.environ, PYTHONIOENCODING="utf-8"))
    print()
    print(r.stdout.strip() or "(no output)")
    if r.returncode != 0:
        ok = False
        print("FAILED:", r.stderr.strip()[-1200:])

print()
print("CATT ready" if ok else "CATT NOT ready - fix before notebook 01 stage A0b")

Setup is done. Continue to **01_prepare_data.ipynb**.